In [5]:
import psutil
def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running
ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
"Ollama not running. Launch ollama before proceeding."
)
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [6]:
import json
with open("instruction-data-with-response.json", "r") as file:
    test_data = json.load(file)

In [7]:
import requests

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {
            "seed": 123, 
            "temperature": 0,
            "num_ctx": 2048
        },
    }

    with requests.post(url=url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines():
            if not line:
                continue
            response_json = json.loads(line)
            if 'message' in response_json:
                response_data += response_json['message']['content']

    return response_data

In [8]:
result = query_model("hello llama")
print(result)

Hello there! It's not every day I get to chat with a llama! How are you doing today?


In [10]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [11]:
# now lets give score to each response of our model by prompting it to the llama using api call(query_model)

for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )

    print("\nDataset response:")
    print(">>", entry["output"])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n----------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a cheetah.

Score:
>> I'd rate the model response "The car is as fast as a cheetah." an 85 out of 100.

Here's why:

* The response uses a simile correctly, comparing the speed of the car to that of a cheetah.
* The comparison is relevant and makes sense, as both cars and cheetahs are known for their speed.
* The phrase "as fast as" is used consistently with the original instruction.

The only reason I wouldn't give it a perfect score is that lightning is often used as an example of extremely rapid movement in English language, so using a more common or relatable comparison like a cheetah is still a good choice. However, if the goal was to exactly replicate the original response's level of speed and vividness, I might deduct a few points for not using lightning specifically.

----------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumu

In [12]:
from tqdm import tqdm
def generate_model_scores(json_data, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."  
        )

        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue
    return scores

score = generate_model_scores(test_data)
print(sum(score) / len(score))

Scoring entries: 100%|██████████| 110/110 [16:00<00:00,  8.74s/it]

47.53636363636364
